In [4]:
import pandas as pd

df = pd.read_csv("autism_screening.csv")
df.head()


,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,...,gender,ethnicity,jundice,austim,contry_of_res,used_app_before,result,age_desc,relation,Class/ASD
0,1,1,1,1,0,0,1,1,0,0,...,f,White-European,no,no,United States,no,6.0,18 and more,Self,NO
1,1,1,0,1,0,0,0,1,0,1,...,m,Latino,no,yes,Brazil,no,5.0,18 and more,Self,NO
2,1,1,0,1,1,0,1,1,1,1,...,m,Latino,yes,yes,Spain,no,8.0,18 and more,Parent,YES
3,1,1,0,1,0,0,1,1,0,1,...,f,White-European,no,yes,United States,no,6.0,18 and more,Self,NO
4,1,0,0,0,0,0,0,1,0,0,...,f,?,no,no,Egypt,no,2.0,18 and more,?,NO


In [5]:
df.columns.tolist()

['A1_Score',
 'A2_Score',
 'A3_Score',
 'A4_Score',
 'A5_Score',
 'A6_Score',
 'A7_Score',
 'A8_Score',
 'A9_Score',
 'A10_Score',
 'age',
 'gender',
 'ethnicity',
 'jundice',
 'austim',
 'contry_of_res',
 'used_app_before',
 'result',
 'age_desc',
 'relation',
 'Class/ASD']

In [6]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Replace missing value markers with NaN
df = df.replace('?', np.nan)

# Drop rows with missing age
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df = df.dropna(subset=['age'])

# Encode categorical columns
cat_cols = ['gender', 'ethnicity', 'jundice', 'austim', 
            'contry_of_res', 'used_app_before', 'relation']

le_dict = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    le_dict[col] = le

# Set up features and target
y = df['Class/ASD'].map({'YES': 1, 'NO': 0, 'Yes': 1, 'No': 0})
X = df.drop(columns=['Class/ASD', 'age_desc', 'result'])

print(X.shape, y.shape)
X.head()

(702, 18) (702,)


,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,age,gender,ethnicity,jundice,austim,contry_of_res,used_app_before,relation
0,1,1,1,1,0,0,1,1,0,0,26.0,0,9,0,0,64,0,4
1,1,1,0,1,0,0,0,1,0,1,24.0,1,3,0,1,13,0,4
2,1,1,0,1,1,0,1,1,1,1,27.0,1,3,1,1,56,0,2
3,1,1,0,1,0,0,1,1,0,1,35.0,0,9,0,1,64,0,4
4,1,0,0,0,0,0,0,1,0,0,40.0,0,11,0,0,22,0,5


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

if y.isna().any():
    raise ValueError("The target column contains unmapped values.")

y = y.astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight="balanced"
)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print(f"Training rows: {len(X_train)}")
print(f"Test rows: {len(X_test)}")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["NO", "YES"]))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))

Training rows: 561
Test rows: 141
Accuracy: 0.950

Classification report:
              precision    recall  f1-score   support

          NO       0.98      0.95      0.97       103
         YES       0.88      0.95      0.91        38

    accuracy                           0.95       141
   macro avg       0.93      0.95      0.94       141
weighted avg       0.95      0.95      0.95       141

Confusion matrix:
[[98  5]
 [ 2 36]]


In [8]:
import joblib

artifact = {
    "model": model,
    "label_encoders": le_dict,
    "feature_columns": X.columns.tolist(),
}
joblib.dump(artifact, "autism_screening_model.joblib")
print("Saved autism_screening_model.joblib")

Saved autism_screening_model.joblib


In [10]:
for col in ['gender', 'ethnicity', 'jundice', 'austim', 'contry_of_res', 'used_app_before', 'relation']:
    print(col, dict(zip(le_dict[col].classes_, le_dict[col].transform(le_dict[col].classes_))))

gender {'f': np.int64(0), 'm': np.int64(1)}
ethnicity {'Asian': np.int64(0), 'Black': np.int64(1), 'Hispanic': np.int64(2), 'Latino': np.int64(3), 'Middle Eastern ': np.int64(4), 'Others': np.int64(5), 'Pasifika': np.int64(6), 'South Asian': np.int64(7), 'Turkish': np.int64(8), 'White-European': np.int64(9), 'others': np.int64(10), nan: np.int64(11)}
jundice {'no': np.int64(0), 'yes': np.int64(1)}
austim {'no': np.int64(0), 'yes': np.int64(1)}
contry_of_res {'Afghanistan': np.int64(0), 'AmericanSamoa': np.int64(1), 'Angola': np.int64(2), 'Argentina': np.int64(3), 'Armenia': np.int64(4), 'Aruba': np.int64(5), 'Australia': np.int64(6), 'Austria': np.int64(7), 'Azerbaijan': np.int64(8), 'Bahamas': np.int64(9), 'Bangladesh': np.int64(10), 'Belgium': np.int64(11), 'Bolivia': np.int64(12), 'Brazil': np.int64(13), 'Burundi': np.int64(14), 'Canada': np.int64(15), 'Chile': np.int64(16), 'China': np.int64(17), 'Costa Rica': np.int64(18), 'Cyprus': np.int64(19), 'Czech Republic': np.int64(20), 'E

In [11]:
print(dict(zip(le_dict['relation'].classes_, le_dict['relation'].transform(le_dict['relation'].classes_))))

{'Health care professional': np.int64(0), 'Others': np.int64(1), 'Parent': np.int64(2), 'Relative': np.int64(3), 'Self': np.int64(4), nan: np.int64(5)}


In [14]:
saved = joblib.load("autism_screening_model.joblib")
print(type(saved))
if isinstance(saved, dict):
    print(saved.keys())

<class 'dict'>
dict_keys(['model', 'label_encoders', 'feature_columns'])
